# Euclid DR1 images: example

Colour images of Euclid DR1 MER mosaics (VIS, Y, J and H, combined by `eummy`), with `streameucliddes.images.euclid_images`. The MER mosaics are 0.1″ per pixel: a stream 5 deg long would be ~180,000 pixels, in tiles of ~1.5 GB per band. So the images are cutouts of a few arcmin at chosen positions, from the ESA Euclid archive (`CUTOUT_SOURCE = "esa"`: cut on the server, ~20 s per band) or from IRSA (`"irsa"`: only the pixels of the cutout read, ~50 s per band); the same mosaics under the same file names, so use the other one when one is down. They are saved in `IMAGE_DIR` (ignored by git), one folder per position, and a filled folder is not downloaded again.

Needs `eummy` (`pipx install eummy`, or its own venv linked at `~/.local/bin/eummy`) and `pyvo` (ESA) or `astroquery` (IRSA).

Sections:
1. one image at a sky position;
2. a stream candidate: its regions and its stars in the match filter (the inputs of `candidate_investigation_example.ipynb`);
3. positions along its track and in its off region;
4. images at these positions, with the stars circled;
5. stamps centred on the stars in the match filter on the track: are the stars counted in the stream real stars, or a feature of the data?

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.images.euclid_images import (
    candidate_image_positions,
    cutout_wcs,
    make_color_image,
    match_filter_star_positions,
    mer_cutouts_for_eummy,
    plot_color_image,
    read_color_image,
)

IMAGE_DIR = "../figures/outputs/images/euclid"
IMAGE_SIZE_ARCMIN = 3.0  # side of each image
CUTOUT_SOURCE = "esa"  # or "irsa": the archive of the cutouts

## 1. One position

VIS, Y, J and H cutouts of the MER tile closest to (ra, dec) (`mer_cutouts_for_eummy`), the colour image (`make_color_image`, `read_color_image`) and its sky coordinates (`cutout_wcs`, `plot_color_image`).

In [ ]:
RA, DEC = 45.341, -45.496  # deg
directory = f"{IMAGE_DIR}/{RA:.4f}_{DEC:+.4f}"
mer_cutouts_for_eummy(RA, DEC, IMAGE_SIZE_ARCMIN, directory, source=CUTOUT_SOURCE)
image = read_color_image(make_color_image(directory))
fig, ax = plot_color_image(image, cutout_wcs(directory), title=f"{IMAGE_SIZE_ARCMIN:g}′ around ({RA:.3f}, {DEC:.3f})")

## 2. Stream candidate

The inputs chosen in `candidate_investigation_example.ipynb`, and only the steps the images need: the stars of the window (`prepare`), the width (`fit_width`) and the regions (`define_regions`). No sky map nor scan of distance moduli.

In [ ]:
NAME = "euclid_dr1_south_candidate"
ENDPOINT_1 = (45.5, -45)  # (ra, dec) deg
ENDPOINT_2 = (44., -51)
CATALOG = "euclid_dr1"
STAR_SELECTION = "morphology_cuts"  # or "xgboost_classifier"
DISTANCE_MODULUS = 17.0
# The match filter, isochrone and width fit: the defaults (get_match_filter(), 13 Gyr and [Fe/H] = -1.9 in the
# default isochrone model, a linear background)
cand = CandidateInvestigation(NAME, ENDPOINT_1, ENDPOINT_2, distance_modulus=DISTANCE_MODULUS)

In [ ]:
%%time
LOAD_MARGIN_DEG = 0.5  # the (ra, dec) box of the window, plus this margin
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
catalog = get_euclid_stars(
    CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION),
    ra_range=((ra_min - LOAD_MARGIN_DEG) % 360.0, (ra_max + LOAD_MARGIN_DEG) % 360.0),
    dec_range=(max(dec_min - LOAD_MARGIN_DEG, -90.0), min(dec_max + LOAD_MARGIN_DEG, 90.0)),
)
cand.prepare(catalog).fit_width().define_regions()
summary = cand.summary()
print(f"phi2_centre = {summary['phi2_centre']:.3f} deg, sigma = {summary['sigma']:.3f} deg, "
      f"excess {summary['n_excess']:.0f} ± {summary['n_excess_error']:.0f} stars in the match filter")

## 3. Positions

Every `IMAGE_STEP_DEG` along phi1 over the regions, on the track and in the off region above at the same phi1 (`candidate_image_positions`).

In [ ]:
IMAGE_STEP_DEG = 1.0
positions = candidate_image_positions(cand, IMAGE_STEP_DEG, off_region="above")
positions

## 4. Images

One image per position; the prepared stars are circled, those in the match filter at the distance modulus in another colour. Do the stars counted in the stream look like point sources? Are there artefacts (bright-star halos, ghosts, tile edges)? Compare with the off images.

In [ ]:
%%time
stars = cand.search.stars
in_filter = cand.stars_in_match_filter()
marked_stars = {
    "stars": (stars["ra"][~in_filter], stars["dec"][~in_filter]),
    "in the match filter": (stars["ra"][in_filter], stars["dec"][in_filter]),
}
for row in positions.itertuples():
    directory = f"{IMAGE_DIR}/{NAME}/{row.region}_phi1_{row.phi1:+.2f}"
    mer_cutouts_for_eummy(row.ra, row.dec, IMAGE_SIZE_ARCMIN, directory, source=CUTOUT_SOURCE)
    image = read_color_image(make_color_image(directory))
    fig, ax = plot_color_image(
        image, cutout_wcs(directory), marked_stars=marked_stars,
        title=f"{row.region}: phi1 = {row.phi1:+.2f}, phi2 = {row.phi2:+.2f} deg, (ra, dec) = ({row.ra:.3f}, {row.dec:.3f})",
    )

## 5. Stamps of the stars in the match filter, on the track

The stars counted in the stream: `N_STAMPS` stars of the on region in the match filter at the distance modulus, spread evenly along phi1 (`match_filter_star_positions`: every k-th one in phi1 order), and a stamp of `STAMP_SIZE_ARCMIN` centred on each. About 80 % of them are field stars (excess 254 of 1258 on DR1), so most stamps show ordinary stars; a feature of the data would show as many centres that are not point sources (artefacts, compact galaxies, edges or holes of the image). The centre is the star; the other stars in the match filter are circled in blue, the other catalog stars in orange.

In [ ]:
N_STAMPS = 24
STAMP_SIZE_ARCMIN = 0.5  # 30 arcsec
targets = match_filter_star_positions(cand, N_STAMPS)
targets[["phi1", "phi2", "ra", "dec", "mag", "color"]].round(3)

In [ ]:
%%time
stars = cand.search.stars
in_filter = cand.stars_in_match_filter()
marked_stars = {
    "stars": (stars["ra"][~in_filter], stars["dec"][~in_filter]),
    "in the match filter": (stars["ra"][in_filter], stars["dec"][in_filter]),
}
n_columns = 6
n_rows = int(np.ceil(len(targets) / n_columns))
fig = plt.figure(figsize=(2.8 * n_columns, 3.0 * n_rows), layout="constrained")
for number, row in enumerate(targets.itertuples(), 1):
    directory = f"{IMAGE_DIR}/{NAME}/match_filter_stars/star_{number:02d}_phi1_{row.phi1:+.3f}"
    mer_cutouts_for_eummy(row.ra, row.dec, STAMP_SIZE_ARCMIN, directory, source=CUTOUT_SOURCE)
    wcs = cutout_wcs(directory)
    ax = fig.add_subplot(n_rows, n_columns, number, projection=wcs)
    plot_color_image(read_color_image(make_color_image(directory)), wcs, marked_stars=marked_stars, ax=ax,
                     legend=False, title=f"{number}: phi1 {row.phi1:+.2f}, VIS {row.mag:.1f}, VIS-H {row.color:.2f}")
    for coord in ax.coords:  # small stamps: no coordinates
        coord.set_ticklabel_visible(False)
        coord.set_axislabel("")
fig.savefig(f"{IMAGE_DIR}/{NAME}/match_filter_stars.png", dpi=120)